# [skip-ci]

# ViralScan Basic Usage

This vignette walks through a complete ViralScan run from raw FASTQ files to the main result tables, HTML report, plots, and AnnData object.

## What we will do

1. Verify that `viralscan`, `kb`, and `snakemake` are available.
2. Download a small public FASTQ pair, or point the notebook at your own FASTQ files.
3. Run ViralScan in NCBI-accession mode using Influenza A segment 8 (`NC_002021.3`) as the viral reference.
4. Locate the sample-specific output directory that ViralScan creates.
5. Inspect the output tables with `pandas` and load the AnnData result with `scanpy`.

## Prerequisites

- ViralScan installed. The conda environment from `environment.yml` is recommended because it includes `kb-python` and `snakemake`.
- `kb`, `snakemake`, and `viralscan` available on `PATH`.
- SRA Toolkit (`prefetch`, `fasterq-dump`) for the example download, or your own paired FASTQ files.
- `NCBI_EMAIL` set in the environment, or an email supplied in the command cell below.

**Estimated runtime:** about 10 minutes on 6 cores for the public example, depending on download speed and machine load.

In [ ]:
# Verify the command-line tools are reachable
!viralscan --help
!kb --version
!snakemake --version

## Step 1 - Prepare a FASTQ pair

The example uses SRR20710651, a public 10x Genomics v3 PBMC library. The SRA Toolkit must be installed; see https://github.com/ncbi/sra-tools/wiki.

To use your own data, skip the download cell and set `r1_path` and `r2_path` in Step 2 to your FASTQ files. ViralScan expects R1 to contain the cell barcode/UMI read for standard 10x libraries and R2 to contain the cDNA read.

In [ ]:
import subprocess
from pathlib import Path

SRR = "SRR20710651"
fastq_dir = Path("fastq") / SRR
fastq_dir.mkdir(parents=True, exist_ok=True)

r1 = fastq_dir / f"{SRR}_1.fastq"
r2 = fastq_dir / f"{SRR}_2.fastq"

if not r1.exists() or not r2.exists():
    subprocess.run(["prefetch", SRR, "--output-directory", str(fastq_dir.parent)], check=True)
    subprocess.run(
        ["fasterq-dump", str(fastq_dir.parent / SRR), "--outdir", str(fastq_dir), "--split-files"],
        check=True,
    )

print("FASTQ files:")
for f in sorted(fastq_dir.glob("*.fastq")):
    size_mb = f.stat().st_size / 1e6
    print(f"  {f.name}  ({size_mb:.1f} MB)")

## Step 2 - Run ViralScan in NCBI accession mode

The command below uses Influenza A segment 8 (`NC_002021.3`) as the viral reference. ViralScan will download the reference from NCBI, build a kallisto index, and quantify the sample.

NCBI requires a contact email. Set `NCBI_EMAIL` in your shell before starting Jupyter, or replace `NCBI_EMAIL_FALLBACK` with your email address.

In [ ]:
import os

output_root = Path("viralscan_output")
r1_path = fastq_dir / f"{SRR}_1.fastq"
r2_path = fastq_dir / f"{SRR}_2.fastq"
NCBI_EMAIL_FALLBACK = None  # e.g. "you@example.org"

cmd = [
    "viralscan",
    "--ncbi-accession",
    "NC_002021.3",
    "--output",
    str(output_root),
    "--sample1",
    str(r1_path),
    "--sample2",
    str(r2_path),
    "--umap",
    "--visual",
    "--cores",
    "6",
]

if not os.environ.get("NCBI_EMAIL"):
    if NCBI_EMAIL_FALLBACK is None:
        raise RuntimeError("Set NCBI_EMAIL or fill NCBI_EMAIL_FALLBACK before running this cell.")
    cmd.extend(["--ncbi-email", NCBI_EMAIL_FALLBACK])

print("Running:", " ".join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])
    raise RuntimeError(f"viralscan exited with code {result.returncode}")
print("viralscan completed successfully.")

## Step 3 - Locate the sample run directory

ViralScan creates one subdirectory per FASTQ pair under `--output`. The directory name is inferred from the R1 filename before the first underscore. For `SRR20710651_1.fastq`, the run directory is `viralscan_output/SRR20710651/`.

In [ ]:
sample_name = r1_path.name.split("_")[0]
sample_dir = output_root / sample_name
results_dir = sample_dir / "results"
plots_dir = sample_dir / "plots"
adata_path = sample_dir / "kb-python" / "counts_unfiltered" / "adata_multimap.h5ad"

print("Sample directory:", sample_dir.resolve())
print("Results directory exists:", results_dir.exists())
print("AnnData exists:", adata_path.exists())

## Step 4 - Inspect the viral summary table

`viral_summary.tsv` has one row per detected virus. If the table is empty or absent, inspect `summary.txt` and the Snakemake logs in the sample directory.

In [ ]:
import pandas as pd

summary_path = results_dir / "viral_summary.tsv"
summary = pd.read_csv(summary_path, sep="\t")
print(f"Rows: {len(summary)}  Columns: {list(summary.columns)}")
summary.head(10)

In [ ]:
per_cell_path = results_dir / "per_cell_viral.tsv"
per_cell = pd.read_csv(per_cell_path, sep="\t")
print(f"Rows: {len(per_cell)}")
per_cell.head(5)

## Step 5 - Load the AnnData result with scanpy

The multimapping-corrected object is written by the workflow under `kb-python/counts_unfiltered/` in the sample run directory.

In [ ]:
import scanpy as sc

adata = sc.read_h5ad(adata_path)
print(adata)
print("\nobs columns:", list(adata.obs.columns))
adata.obs.head(5)

## Step 6 - Check the report and plots

`report.html` is written at the sample run directory root. Static PNG plots and optional UMAP HTML files are written under `plots/`.

In [ ]:
report_path = sample_dir / "report.html"
print(f"HTML report: {report_path.resolve()}")
print(f"Report exists: {report_path.exists()}")
if report_path.exists():
    print(f"Report size: {report_path.stat().st_size / 1024:.1f} KB")

plots = sorted(plots_dir.glob("*.png")) if plots_dir.exists() else []
print(f"\nPNG plots generated ({len(plots)}):")
for p in plots:
    print(f"  {p.name}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

if plots:
    img = mpimg.imread(plots[0])
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.imshow(img)
    ax.axis("off")
    ax.set_title(plots[0].stem)
    plt.tight_layout()
    plt.show()
else:
    print(
        "No PNG plots found. This can happen when visual output is disabled or no virus is detected."
    )

## Summary

In this vignette we:

- Prepared a public 10x PBMC FASTQ pair, with notes for substituting your own data.
- Used ViralScan's NCBI accession mode to fetch the Influenza A reference (`NC_002021.3`), build a kallisto index, and quantify viral load.
- Located the sample-specific output directory under `--output`.
- Inspected `results/viral_summary.tsv` and `results/per_cell_viral.tsv` with pandas.
- Loaded `kb-python/counts_unfiltered/adata_multimap.h5ad` with scanpy.
- Located `report.html` and saved plots.

**Next step:** see `cell_type_enrichment.ipynb` to stratify viral prevalence by cell type.